# 🧠 Cardivore AI — Demo Notebook

Welcome to the **Cardivore AI Demo**.  
This notebook demonstrates how to build and explore the Pokémon card ROI dataset.

- **01_pipeline_builder.ipynb** → Build the local demo database  
- **02_demo_launcher.ipynb** → View the ROI Dashboard  

> 💡 Tip: If `/data/database/card_demo.db` does not exist, run `01_pipeline_builder.ipynb` first.


In [ ]:
# --- Cell 0: Bootstrap Environment ---
import os, sys
from importlib import reload

# Determine the project root (one directory up from notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(".."))

# Ensure it's in the path so imports work
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Auto-reload local utils when edited
%load_ext autoreload
%autoreload 2

print(f"✅ Environment bootstrapped.\nPROJECT_ROOT: {PROJECT_ROOT}")




In [ ]:
# --- Cell 1: Build All Tables (Demo) ---

import os
import pandas as pd
from cardivore_ai.utils.io_utils import normalize_market_movers_df
from cardivore_ai.utils.db_utils import get_engine, create_table_if_not_exists

# --- Setup ---
engine = get_engine(demo_mode=True)
DATA_DIR = os.path.join(PROJECT_ROOT,  "data", "raw")

# --- Read CSVs ---
raw_path = os.path.join(DATA_DIR, "raw_historic_sales.csv")
psa_path = os.path.join(DATA_DIR, "psa10_historic_sales.csv")

raw_df = pd.read_csv(raw_path)
psa_df = pd.read_csv(psa_path)

# --- Normalize / clean using your universal function ---
raw_df = normalize_market_movers_df(raw_df)
psa_df = normalize_market_movers_df(psa_df)

print("✅ Cleaned numeric fields and standardized columns:")
display(raw_df.head(3))

# --- Create or replace tables in SQLite ---
create_table_if_not_exists(engine, raw_df, "psa10_raw_sales")
create_table_if_not_exists(engine, psa_df, "psa10_historic_sales")

raw_df.to_sql("psa10_raw_sales", engine, if_exists="replace", index=False)
psa_df.to_sql("psa10_historic_sales", engine, if_exists="replace", index=False)

print("✅ Tables created and loaded: psa10_raw_sales, psa10_historic_sales")



In [ ]:
# --- Cell 2: Build or Load card_summary ---
from cardivore_ai.utils.pipeline_helper import get_summary_df
from cardivore_ai.utils.fe_helpers import display_roi_filter

summary_df, engine = get_summary_df()
display_roi_filter(summary_df)
